# 创建部署

让我们为模块 5 中创建的 `task_maistro` 应用创建一个部署。

## 代码结构

创建 LangGraph Platform 部署需要[提供以下信息](https://docs.langchain.com/langsmith/application-structure)：

* [LangGraph API 配置文件](https://docs.langchain.com/langsmith/application-structure#configuration-file) - `langgraph.json`
* 实现应用逻辑的图 - 例如 `task_maistro.py`
* 指定运行应用所需依赖的文件 - `requirements.txt`
* 提供应用运行所需的环境变量 - `.env` 或 `docker-compose.yml`

这些已经存在于 `module-6/deployment` 目录中！

## CLI

[LangGraph CLI](https://docs.langchain.com/langsmith/cli) 是用于创建 LangGraph Platform 部署的命令行接口。

In [ ]:
%%capture --no-stderr
%pip install -U langgraph-cli

要创建 <!-- [~self-hosted deployment~](https://langchain-ai.github.io/langgraph/how-tos/deploy-self-hosted/#how-to-do-a-self-hosted-deployment-of-langgraph) --> [自托管部署](https://docs.langchain.com/langsmith/self_hosted_data_plane)，我们需要执行几个步骤。

### 为 LangGraph Server 构建 Docker 镜像

首先使用 langgraph CLI 为 [LangGraph Server](https://docs.google.com/presentation/d/18MwIaNR2m4Oba6roK_2VQcBE_8Jq_SI7VHTXJdl7raU/edit#slide=id.g313fb160676_0_32) 创建 Docker 镜像。

这将把我们的图和依赖打包成一个 Docker 镜像。

Docker 镜像是 Docker 容器的模板，包含运行应用所需的代码和依赖。

确保已安装 [Docker](https://docs.docker.com/engine/install/)，然后运行以下命令创建 Docker 镜像 `my-image`：

```
$ cd module-6/deployment
$ langgraph build -t my-image
```

### 设置 Redis 和 PostgreSQL

如果你已经有 Redis 和 PostgreSQL 在运行（例如本地或其他服务器），则可以[单独](https://docs.langchain.com/langsmith/deploy-hybrid#running-the-application-locally)创建和运行 LangGraph Server 容器，并使用 Redis 和 PostgreSQL 的 URI：

```
docker run \
    --env-file .env \
    -p 8123:8000 \
    -e REDIS_URI="foo" \
    -e DATABASE_URI="bar" \
    -e LANGSMITH_API_KEY="baz" \
    my-image
```

或者，你可以使用提供的 `docker-compose.yml` 文件根据定义的服务创建三个独立容器：

* `langgraph-redis`：使用官方 Redis 镜像创建新容器。
* `langgraph-postgres`：使用官方 Postgres 镜像创建新容器。
* `langgraph-api`：使用你预先构建的镜像创建新容器。

只需复制 `docker-compose-example.yml` 并添加以下环境变量来运行已部署的 `task_maistro` 应用：

* `IMAGE_NAME`（例如 `my-image`）
* `LANGSMITH_API_KEY`
* `DEEPSEEK_API_KEY`

然后，<!-- [~launch the deployment~](https://langchain-ai.github.io/langgraph/how-tos/deploy-self-hosted/#using-docker-compose) [launch the deployment](https://docs.langchain.com/langsmith/self_hosted_data_plane): --> 启动部署！

```
$ cd module-6/deployment
$ docker compose up
```